In [1]:
#new eficiant  more than

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import (
    Dense,
    GlobalAveragePooling2D,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    Callback
)
import matplotlib.pyplot as plt
import os
import time

# ==================================================
# GPU CHECK
# ==================================================
print("=" * 60)
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("=" * 60)

# ==================================================
# DATASET PATH
# ==================================================
DATASET_PATH = "Datasets/train"

# ==================================================
# IMAGE GENERATOR
# ==================================================
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    horizontal_flip=True,
    fill_mode='nearest'
)

# ==================================================
# TRAIN DATA
# ==================================================
train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# ==================================================
# VALIDATION DATA
# ==================================================
val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print("\nDATASET INFO")
print("=" * 60)
print("Classes      :", train_data.num_classes)
print("Train Images :", train_data.samples)
print("Val Images   :", val_data.samples)
print("=" * 60)

# ==================================================
# EFFICIENTNETB0
# ==================================================
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

# ==================================================
# CUSTOM HEAD
# ==================================================
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)

x = Dense(
    512,
    activation='relu'
)(x)

x = Dropout(0.5)(x)

x = Dense(
    256,
    activation='relu'
)(x)

x = Dropout(0.3)(x)

outputs = Dense(
    train_data.num_classes,
    activation='softmax'
)(x)

model = Model(
    inputs=base_model.input,
    outputs=outputs
)

# ==================================================
# LOSS FUNCTION
# ==================================================
loss_fn = tf.keras.losses.CategoricalCrossentropy(
    label_smoothing=0.1
)

# ==================================================
# COMPILE
# ==================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=loss_fn,
    metrics=['accuracy']
)

model.summary()

# ==================================================
# CALLBACK
# ==================================================
class ProgressCallback(Callback):

    def on_epoch_begin(self, epoch, logs=None):
        print(f"\nStarting Epoch {epoch+1}")

    def on_epoch_end(self, epoch, logs=None):
        print(
            f"Epoch {epoch+1} | "
            f"Train Acc: {logs['accuracy']:.4f} | "
            f"Val Acc: {logs['val_accuracy']:.4f}"
        )

# ==================================================
# CALLBACKS
# ==================================================
os.makedirs("models", exist_ok=True)

callbacks = [

    ProgressCallback(),

    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        verbose=1
    ),

    ModelCheckpoint(
        "newmodels/best_model.keras",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# ==================================================
# PHASE 1 TRAINING
# ==================================================
print("\nPHASE 1 TRAINING")

start_time = time.time()

history1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=11,
    callbacks=callbacks
)

# ==================================================
# PHASE 2 FINE TUNING
# ==================================================
print("\nFINE TUNING STARTED")

base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=loss_fn,
    metrics=['accuracy']
)

history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)

# ==================================================
# SAVE MODEL
# ==================================================
model.save(
    "newmodels/plant_disease_model.keras"
)

end_time = time.time()

print("\nTRAINING COMPLETED")
print(
    f"Training Time: {(end_time-start_time)/60:.2f} Minutes"
)

# ==================================================
# COMBINE HISTORY
# ==================================================
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']

loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

# ==================================================
# ACCURACY GRAPH
# ==================================================
plt.figure(figsize=(10,5))

plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')

plt.title("Plant Disease Detection Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()

plt.show()

# ==================================================
# LOSS GRAPH
# ==================================================
plt.figure(figsize=(10,5))

plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')

plt.title("Plant Disease Detection Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()

plt.show()

# ==================================================
# FINAL RESULT
# ==================================================
print("\nFINAL RESULTS")
print("=" * 60)

print(
    f"Best Training Accuracy   : "
    f"{max(acc)*100:.2f}%"
)

print(
    f"Best Validation Accuracy : "
    f"{max(val_acc)*100:.2f}%"
)

print("=" * 60)
print("MODEL SAVED:")
print("newmodels/plant_disease_model.keras")
print("=" * 60)

TensorFlow Version: 2.21.0
GPU Available: []
Found 56251 images belonging to 38 classes.
Found 14044 images belonging to 38 classes.

DATASET INFO
Classes      : 38
Train Images : 56251
Val Images   : 14044
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,851,657 (18.51 MB)

 Trainable params: 799,526 (3.05 MB)

 Non-trainable params: 4,052,131 (15.46 MB)


PHASE 1 TRAINING

Starting Epoch 1
Epoch 1/10
1758/1758 ━━━━━━━━━━━━━━━━━━━━ 0s 939ms/step - accuracy: 0.6807 - loss: 1.7423Epoch 1 | Train Acc: 0.7948 | Val Acc: 0.9428

Epoch 1: val_accuracy improved from None to 0.94282, saving model to newmodels/best_model.keras
1758/1758 ━━━━━━━━━━━━━━━━━━━━ 2053s 1s/step - accuracy: 0.7948 - loss: 1.4159 - val_accuracy: 0.9428 - val_loss: 0.9600 - learning_rate: 0.0010

Starting Epoch 2
Epoch 2/10
1758/1758 ━━━━━━━━━━━━━━━━━━━━ 0s 934ms/step - accuracy: 0.8915 - loss: 1.1500Epoch 2 | Train Acc: 0.8976 | Val Acc: 0.9526

Epoch 2: val_accuracy improved from 0.94282 to 0.95258, saving model to newmodels/best_model.keras
1758/1758 ━━━━━━━━━━━━━━━━━━━━ 2054s 1s/step - accuracy: 0.8976 - loss: 1.1298 - val_accuracy: 0.9526 - val_loss: 0.9207 - learning_rate: 0.0010

Starting Epoch 3
Epoch 3/10
1758/1758 ━━━━━━━━━━━━━━━━━━━━ 0s 962ms/step - accuracy: 0.9131 - loss: 1.0824Epoch 3 | Train Acc: 0.9157 | Val Acc: 0.9631

Epoch 3: val_accuracy improved from

KeyboardInterrupt: 

accuracy 97%

from tensorflow.keras.models import load_model

model = load_model("newmodels/best_model.keras")

print("Best model loaded successfully!")

In [9]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

# Load Best Model
model = load_model("newmodels/best_model.keras")

print("Best model loaded successfully!")

# Test Dataset Path
TEST_PATH = "Datasets/test"

# Test Data Generator
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# Test Data
test_data = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Evaluate
test_loss, test_acc = model.evaluate(test_data)

print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

Best model loaded successfully!
Found 23 images belonging to 6 classes.


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 6), output.shape=(None, 38)

failed abouv

In [7]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input
import numpy as np
import os

model = load_model("newmodels/best_model.keras")

# IMPORTANT:
# Replace with your actual class order from training
class_names = list(train_data.class_indices.keys())

test_folder = "Datasets/test"

for file in os.listdir(test_folder):

    if file.lower().endswith(('.jpg', '.jpeg', '.png')):

        path = os.path.join(test_folder, file)

        img = image.load_img(path, target_size=(224,224))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)

        pred = model.predict(img, verbose=0)

        predicted = class_names[np.argmax(pred)]
        confidence = np.max(pred) * 100

        print(f"\nFile       : {file}")
        print(f"Prediction : {predicted}")
        print(f"Confidence : {confidence:.2f}%")


File       : AppleCedarRust1.JPG
Prediction : Apple___Cedar_apple_rust
Confidence : 98.84%

File       : AppleCedarRust2.JPG
Prediction : Apple___Cedar_apple_rust
Confidence : 86.33%

File       : AppleCedarRust3.JPG
Prediction : Apple___Cedar_apple_rust
Confidence : 95.09%

File       : AppleCedarRust4.JPG
Prediction : Apple___Cedar_apple_rust
Confidence : 95.68%

File       : AppleScab1.JPG
Prediction : Apple___Apple_scab
Confidence : 78.70%

File       : AppleScab2.JPG
Prediction : Apple___Apple_scab
Confidence : 84.61%

File       : AppleScab3.JPG
Prediction : Apple___Apple_scab
Confidence : 47.15%

File       : CornCommonRust1.JPG
Prediction : Corn_(maize)___Common_rust_
Confidence : 87.40%

File       : CornCommonRust2.JPG
Prediction : Corn_(maize)___Common_rust_
Confidence : 90.46%

File       : CornCommonRust3.JPG
Prediction : Corn_(maize)___Common_rust_
Confidence : 89.09%

File       : PotatoEarlyBlight1.JPG
Prediction : Potato___Early_blight
Confidence : 88.68%

File       

In [8]:
#accuracy est

import os
import shutil

test_dir = "Datasets/test"

mapping = {
    "AppleScab": "Apple___Apple_scab",
    "AppleCedarRust": "Apple___Cedar_apple_rust",
    "CornCommonRust": "Corn_(maize)___Common_rust_",
    "PotatoEarlyBlight": "Potato___Early_blight",
    "PotatoHealthy": "Potato___healthy",
    "TomatoEarlyBlight": "Tomato___Early_blight"
}

for file in os.listdir(test_dir):

    for key, folder_name in mapping.items():

        if file.startswith(key):

            dst_folder = os.path.join(test_dir, folder_name)

            os.makedirs(dst_folder, exist_ok=True)

            shutil.move(
                os.path.join(test_dir, file),
                os.path.join(dst_folder, file)
            )

            break

print("Done!")

Done!


In [6]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input
import numpy as np
import os

model = load_model("newmodels/best_model.keras")

class_names = [
    'Apple___Apple_scab',
    'Apple___Black_rot',
    'Apple___Cedar_apple_rust',
    # add all 38 class names here
]

test_folder = "Datasets/test"

for file in os.listdir(test_folder):

    if file.lower().endswith(('.jpg','.jpeg','.png')):

        img_path = os.path.join(test_folder, file)

        img = image.load_img(img_path, target_size=(224,224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = preprocess_input(img_array)

        pred = model.predict(img_array, verbose=0)

        predicted_class = class_names[np.argmax(pred)]
        confidence = np.max(pred) * 100

        print(f"{file}")
        print(f"Prediction : {predicted_class}")
        print(f"Confidence : {confidence:.2f}%")
        print("-"*50)

AppleCedarRust1.JPG
Prediction : Apple___Cedar_apple_rust
Confidence : 98.84%
--------------------------------------------------
AppleCedarRust2.JPG
Prediction : Apple___Cedar_apple_rust
Confidence : 86.33%
--------------------------------------------------
AppleCedarRust3.JPG
Prediction : Apple___Cedar_apple_rust
Confidence : 95.09%
--------------------------------------------------
AppleCedarRust4.JPG
Prediction : Apple___Cedar_apple_rust
Confidence : 95.68%
--------------------------------------------------
AppleScab1.JPG
Prediction : Apple___Apple_scab
Confidence : 78.70%
--------------------------------------------------
AppleScab2.JPG
Prediction : Apple___Apple_scab
Confidence : 84.61%
--------------------------------------------------
AppleScab3.JPG
Prediction : Apple___Apple_scab
Confidence : 47.15%
--------------------------------------------------


IndexError: list index out of range

In [ ]:
acuracy #  94 percent trained best_model.keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
    Callback
)
import matplotlib.pyplot as plt
import os
import time

# ====================================
# CHECK GPU
# ====================================
print("="*60)
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("="*60)

# ====================================
# DATASET PATH
# ====================================
DATASET_PATH = "Datasets/train"   # Change if needed

# ====================================
# IMAGE PREPROCESSING
# ====================================
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True
)

# ====================================
# TRAIN DATA
# ====================================
train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# ====================================
# VALIDATION DATA
# ====================================
val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print("\n📊 DATASET INFORMATION")
print("="*60)
print("Number of Classes :", train_data.num_classes)
print("Training Images   :", train_data.samples)
print("Validation Images :", val_data.samples)
print("Batch Size        :", train_data.batch_size)
print("="*60)

# ====================================
# LOAD MOBILENETV2
# ====================================
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# ====================================
# CUSTOM HEAD
# ====================================
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)

output = Dense(
    train_data.num_classes,
    activation='softmax'
)(x)

# ====================================
# BUILD MODEL
# ====================================
model = Model(
    inputs=base_model.input,
    outputs=output
)

# ====================================
# COMPILE MODEL
# ====================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ====================================
# MODEL SUMMARY
# ====================================
model.summary()

# ====================================
# CUSTOM PROGRESS CALLBACK
# ====================================
class ProgressCallback(Callback):

    def on_epoch_begin(self, epoch, logs=None):
        print(f"\n🚀 Starting Epoch {epoch+1}")

    def on_epoch_end(self, epoch, logs=None):
        print(
            f"✅ Epoch {epoch+1} Completed | "
            f"Train Acc: {logs['accuracy']:.4f} | "
            f"Val Acc: {logs['val_accuracy']:.4f}"
        )

# ====================================
# CALLBACKS
# ====================================
os.makedirs("models", exist_ok=True)

callbacks = [

    ProgressCallback(),

    EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        verbose=1
    ),

    ModelCheckpoint(
        "models/best_model.keras",
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    )
]

# ====================================
# TRAIN MODEL
# ====================================
print("\n🔥 TRAINING STARTED...\n")

start_time = time.time()

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

end_time = time.time()

# ====================================
# SAVE MODEL
# ====================================
model.save(
    "models/plant_disease_model.keras"
)

print("\n✅ TRAINING COMPLETED")
print(
    f"⏱ Training Time: {(end_time-start_time)/60:.2f} Minutes"
)

# ====================================
# ACCURACY GRAPH
# ====================================
plt.figure(figsize=(10,5))

plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title("Plant Disease Detection Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

# ====================================
# FINAL ACCURACY
# ====================================
print("\n🎯 FINAL RESULTS")
print(
    f"Training Accuracy   : "
    f"{max(history.history['accuracy'])*100:.2f}%"
)

print(
    f"Validation Accuracy : "
    f"{max(history.history['val_accuracy'])*100:.2f}%"
)

print("\n💾 Model Saved:")
print("models/plant_disease_model.keras")

TensorFlow Version: 2.21.0
GPU Available: []
Found 56251 images belonging to 38 classes.
Found 14044 images belonging to 38 classes.

📊 DATASET INFORMATION
Number of Classes : 38
Training Images   : 56251
Validation Images : 14044
Batch Size        : 32


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,600,806 (9.92 MB)

 Trainable params: 340,262 (1.30 MB)

 Non-trainable params: 2,260,544 (8.62 MB)


🔥 TRAINING STARTED...


🚀 Starting Epoch 1
Epoch 1/15
1758/1758 ━━━━━━━━━━━━━━━━━━━━ 0s 965ms/step - accuracy: 0.7583 - loss: 0.8514✅ Epoch 1 Completed | Train Acc: 0.8399 | Val Acc: 0.9201

Epoch 1: val_accuracy improved from None to 0.92011, saving model to models/best_model.keras
1758/1758 ━━━━━━━━━━━━━━━━━━━━ 2134s 1s/step - accuracy: 0.8399 - loss: 0.5322 - val_accuracy: 0.9201 - val_loss: 0.2454 - learning_rate: 0.0010

🚀 Starting Epoch 2
Epoch 2/15
1758/1758 ━━━━━━━━━━━━━━━━━━━━ 0s 731ms/step - accuracy: 0.8996 - loss: 0.3171✅ Epoch 2 Completed | Train Acc: 0.9018 | Val Acc: 0.9322

Epoch 2: val_accuracy improved from 0.92011 to 0.93221, saving model to models/best_model.keras
1758/1758 ━━━━━━━━━━━━━━━━━━━━ 1612s 917ms/step - accuracy: 0.9018 - loss: 0.3064 - val_accuracy: 0.9322 - val_loss: 0.2077 - learning_rate: 0.0010

🚀 Starting Epoch 3
Epoch 3/15
1758/1758 ━━━━━━━━━━━━━━━━━━━━ 0s 758ms/step - accuracy: 0.9097 - loss: 0.2806✅ Epoch 3 Completed | Train Acc: 0.9112 | Val Acc

In [5]:
import os
import pickle

dataset_path = "Datasets/train"

class_names = sorted([
    folder for folder in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, folder))
])

with open("models/class_names.pkl", "wb") as f:
    pickle.dump(class_names, f)

print("✅ Class names saved successfully!")
print("Total Classes:", len(class_names))
print(class_names[:10])

✅ Class names saved successfully!
Total Classes: 38
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight']


In [6]:
import tensorflow as tf
import pickle

# Load model
model = tf.keras.models.load_model(
    "models/best_model.keras"
)

# Load classes
with open("models/class_names.pkl", "rb") as f:
    class_names = pickle.load(f)

print("✅ Model Loaded")
print("✅ Classes Loaded")
print("Total Classes:", len(class_names))
print("Output Shape:", model.output_shape)

✅ Model Loaded
✅ Classes Loaded
Total Classes: 38
Output Shape: (None, 38)


In [ ]:
#
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
    Callback
)
import matplotlib.pyplot as plt
import os
import time

# ====================================
# CHECK GPU
# ====================================
print("="*60)
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("="*60)

# ====================================
# DATASET PATH
# ====================================
DATASET_PATH = "Datasets/train"   # Change if needed

# ====================================
# IMAGE PREPROCESSING
# ====================================
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True
)

# ====================================
# TRAIN DATA
# ====================================
train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# ====================================
# VALIDATION DATA
# ====================================
val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print("\n📊 DATASET INFORMATION")
print("="*60)
print("Number of Classes :", train_data.num_classes)
print("Training Images   :", train_data.samples)
print("Validation Images :", val_data.samples)
print("Batch Size        :", train_data.batch_size)
print("="*60)

# ====================================
# LOAD MOBILENETV2
# ====================================
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# ====================================
# CUSTOM HEAD
# ====================================
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)

output = Dense(
    train_data.num_classes,
    activation='softmax'
)(x)

# ====================================
# BUILD MODEL
# ====================================
model = Model(
    inputs=base_model.input,
    outputs=output
)

# ====================================
# COMPILE MODEL
# ====================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ====================================
# MODEL SUMMARY
# ====================================
model.summary()

# ====================================
# CUSTOM PROGRESS CALLBACK
# ====================================
class ProgressCallback(Callback):

    def on_epoch_begin(self, epoch, logs=None):
        print(f"\n🚀 Starting Epoch {epoch+1}")

    def on_epoch_end(self, epoch, logs=None):
        print(
            f"✅ Epoch {epoch+1} Completed | "
            f"Train Acc: {logs['accuracy']:.4f} | "
            f"Val Acc: {logs['val_accuracy']:.4f}"
        )

# ====================================
# CALLBACKS
# ====================================
os.makedirs("models", exist_ok=True)

callbacks = [

    ProgressCallback(),

    EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        verbose=1
    ),

    ModelCheckpoint(
        "models/best_model.keras",
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    )
]

# ====================================
# TRAIN MODEL
# ====================================
print("\n🔥 TRAINING STARTED...\n")

start_time = time.time()

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

end_time = time.time()

# ====================================
# SAVE MODEL
# ====================================
model.save(
    "models/plant_disease_model.keras"
)

print("\n✅ TRAINING COMPLETED")
print(
    f"⏱ Training Time: {(end_time-start_time)/60:.2f} Minutes"
)

# ====================================
# ACCURACY GRAPH
# ====================================
plt.figure(figsize=(10,5))

plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title("Plant Disease Detection Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

# ====================================
# FINAL ACCURACY
# ====================================
print("\n🎯 FINAL RESULTS")
print(
    f"Training Accuracy   : "
    f"{max(history.history['accuracy'])*100:.2f}%"
)

print(
    f"Validation Accuracy : "
    f"{max(history.history['val_accuracy'])*100:.2f}%"
)

print("\n💾 Model Saved:")
print("models/plant_disease_model.keras")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os

# Dataset Path
DATASET_PATH = "Datasets_train"

# Image Generator
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True
)

# Training Data
train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Validation Data
val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Load MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# Custom Layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)

output = Dense(
    train_data.num_classes,
    activation='softmax'
)(x)

model = Model(
    inputs=base_model.input,
    outputs=output
)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2
    ),

    ModelCheckpoint(
        "models/best_model.keras",
        save_best_only=True,
        monitor='val_accuracy'
    )
]

# Train
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks
)

# Save Model
os.makedirs("models", exist_ok=True)
model.save("models/plant_disease_model.keras")

print("✅ Training Completed")